# Development notebook for converting Geo-encoded polygon mask predictions to JSON friendly ICDAR 2025 Map Text Reading Competition format.

In [ ]:
# Imports
from typing import Final
from os import getenv
from pathlib import Path
from functools import partial
from dotenv import find_dotenv, load_dotenv
from json import load as load_json, dump as dump_json
from shapely import get_coordinates
from rasterio import open as open_raster
from rasterio.transform import AffineTransformer
from geopandas import read_file
from outputs import pixel_ref_geometries

# Constants and presets
PROJECT_DIR: Final[Path]
PROJECT_DIR = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

# File paths to Geo-encoded predictions
geopred_fns = [*LOCAL_DIR.glob("outputs/toponym-extractor/revised-v2/*.gpkg")]

## Handling predictions

In [ ]:
# Select example predictions file and tiff transformer.
eg_idx: int = 7
eg_fn = geopred_fns[eg_idx]

with open_raster(LOCAL_DIR.joinpath(f"data/tiffs/{eg_fn.stem}.tif")) as tif:
    tif_transformer = AffineTransformer(tif.transform)
    tif_crs = tif.read_crs()
# partially fill pixel_ref_transformer
transformer = partial(pixel_ref_geometries, tif_transformer)

geopreds = read_file(eg_fn)

# convert geopred geometries
geopreds = geopreds.to_crs(tif_crs)
geopreds["geometry"] = geopreds.geometry\
    .transform(transformer, include_z = False)
geopreds["geometry"] = geopreds.geometry.exterior

# change column name: word -> text
geopreds.rename(columns = {"word": "text"}, inplace = True)

geopreds.sort_values(["groupid", "wordid"])

In [ ]:
get_coordinates(geopreds.geometry.iloc[0]).tolist()

In [ ]:
order_func = lambda d: dict.get(d, "wordid")

toponyms = []
for group in geopreds.groupid.unique():
    temp = geopreds\
        .loc[(geopreds.groupid == group), ["wordid", "text", "geometry"]]\
        .to_dict(orient = "records")
    toponyms.append(temp)

for temp in range(len(toponyms)):
    # Order words within each toponym
    toponyms[temp] = sorted(toponyms[temp], key = order_func)
    for word in toponyms[temp]:
        # remove wordid
        del word["wordid"]
        # convert shape to list of xy pixel coordinates
        word["vertices"] = get_coordinates(word.pop("geometry")).tolist()

toponyms

## Handling Ground-truths

In [ ]:
gdf = read_file(LOCAL_DIR.joinpath("outputs/manual-labelling/refined/glam-st17ne-2.gpkg"))
gdf.sort_values(["groupid", "wordid"])

In [ ]:
gdf.dtypes

In [ ]:
gdf[gdf.word.str.contains("$", regex = False)]